In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
# --------------------------
# 1) Dataset Wine
# https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html
# https://archive.ics.uci.edu/dataset/109/wine
# --------------------------
data = load_wine()
X, y = data.data, data.target
mask = y != 2           # filtramos clase 2
X, y = X[mask], y[mask] # ahora binario (clases 0 y 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Estandarización
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

# Para perceptrón clásico usamos etiquetas en {-1,+1}
y_train_pm1 = np.where(y_train == 1, 1, -1)
y_test_pm1  = np.where(y_test == 1, 1, -1)


In [ ]:
print(X_train)

[[-0.23655888 -0.25179223 -0.19888842 ... -1.05993134 -1.1052945
  -0.89951312]
 [ 0.4712347  -1.14858802  0.08482006 ... -0.19522705 -2.17962267
  -0.12351168]
 [-1.20692103  2.58252536  0.19121074 ... -1.86287103  1.5695634
  -1.21939158]
 ...
 [-1.0585127  -0.5175095   0.29760142 ... -0.00993327 -0.60101801
  -1.01206295]
 [-0.32788708 -0.6835828  -1.12094097 ... -0.44228542 -2.92507487
  -1.01206295]
 [ 0.85937956 -0.1632198   0.08482006 ...  1.10182938 -0.11866659
   0.58732933]]


In [ ]:
# --------------------------
# 2) Logistic Regression
# --------------------------
clf = LogisticRegression()
clf.fit(X_train, y_train)
yhat_tr_log = clf.predict(X_train)
yhat_te_log = clf.predict(X_test)
acc_tr_log = accuracy_score(y_train, yhat_tr_log) * 100
acc_te_log = accuracy_score(y_test,  yhat_te_log) * 100

print(f"LogisticRegression -> acc_train: {acc_tr_log:.2f}% | acc_test: {acc_te_log:.5f}%")

LogisticRegression -> acc_train: 100.00% | acc_test: 96.96970%


In [ ]:
# --------------------------
# 2) Perceptrón
# --------------------------

# 1. Preparo los datos
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor =torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 2. Defino el modelo
class Perceptron(nn.Module):
    def __init__(self, input_size, p_drop=0):
        super().__init__()
        self.dropout = nn.Dropout(p_drop)
        self.linear = nn.Linear(input_size, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        #x = self.dropout(x)
        x = self.linear(x)
        x = self.activation(x)
        return x

# 3. Inicializo el modelo
input_size = X_train_tensor.shape[1]
model = Perceptron(input_size)
criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification
optimizer = optim.SGD(model.parameters(), lr=0.1) # Stochastic Gradient Descent

# 4. Entreno el modelo
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward() # Calculo los gradientes
    optimizer.step() # Modifico los pesos

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# 5. Evaluo el modelo
with torch.no_grad():
    yhat_tr_perceptron = (model(X_train_tensor) > 0.5).float()
    acc_tr_log = (yhat_tr_perceptron == y_train_tensor).sum().item() / len(y_train_tensor) * 100
    yhat_te_perceptron = (model(X_test_tensor) > 0.5).float()
    acc_te_log = (yhat_te_perceptron == y_test_tensor).sum().item() / len(y_test_tensor) * 100
    print(f"Perceptron (pytorch) -> acc_train: {acc_tr_log:.2f}% | acc_test: {acc_te_log:.5f}%")

Epoch [100/1000], Loss: 0.0819
Epoch [200/1000], Loss: 0.0519
Epoch [300/1000], Loss: 0.0390
Epoch [400/1000], Loss: 0.0317
Epoch [500/1000], Loss: 0.0268
Epoch [600/1000], Loss: 0.0234
Epoch [700/1000], Loss: 0.0208
Epoch [800/1000], Loss: 0.0187
Epoch [900/1000], Loss: 0.0171
Epoch [1000/1000], Loss: 0.0157
Perceptron (pytorch) -> acc_train: 100.00% | acc_test: 96.96970%


Ejercicios:

1) Implemetar la clase MLP3 (perceptrón multicapa de 3 capas densas).
2) Entrenar, evaluar y comparar resutlados para MLP3.